In [2]:
import re
import numpy as np
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import transformers
from transformers import pipeline

In [3]:
data = pd.read_csv('book_reviews_sample.csv')
data.head(5)

,index,reviewText,rating
0,11494,Clean and funny. A bit busy with all the diffe...,3
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4
2,1463,Good thing that this is a free story. I read i...,1
3,10342,"Action, action, action! Equipment keeps gettin...",4
4,5256,this was hands down the worse book i have ever...,1


In [4]:
data['reviewText_lowercase'] = data['reviewText'].str.lower()

In [5]:
data['reviewText_clean'] = data.apply(lambda x: re.sub(r'[^\w\s]','', x['reviewText_lowercase']), axis = 1)

In [6]:
data.head(5)

,index,reviewText,rating,reviewText_lowercase,reviewText_clean
0,11494,Clean and funny. A bit busy with all the diffe...,3,clean and funny. a bit busy with all the diffe...,clean and funny a bit busy with all the differ...
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,alex a sexy hot cop and the phd candidate. wha...,alex a sexy hot cop and the phd candidate what...
2,1463,Good thing that this is a free story. I read i...,1,good thing that this is a free story. i read i...,good thing that this is a free story i read it...
3,10342,"Action, action, action! Equipment keeps gettin...",4,"action, action, action! equipment keeps gettin...",action action action equipment keeps getting b...
4,5256,this was hands down the worse book i have ever...,1,this was hands down the worse book i have ever...,this was hands down the worse book i have ever...


In [7]:
text_sent_analyse = SentimentIntensityAnalyzer()

In [12]:
data['sent_analyze_score'] = data['reviewText_clean'].apply(lambda x: text_sent_analyse.polarity_scores(x)['compound'])

In [13]:
data.head(5)

,index,reviewText,rating,reviewText_lowercase,reviewText_clean,sent_analyze_score
0,11494,Clean and funny. A bit busy with all the diffe...,3,clean and funny. a bit busy with all the diffe...,clean and funny a bit busy with all the differ...,0.7684
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,alex a sexy hot cop and the phd candidate. wha...,alex a sexy hot cop and the phd candidate what...,0.9325
2,1463,Good thing that this is a free story. I read i...,1,good thing that this is a free story. i read i...,good thing that this is a free story i read it...,0.6740
3,10342,"Action, action, action! Equipment keeps gettin...",4,"action, action, action! equipment keeps gettin...",action action action equipment keeps getting b...,0.6652
4,5256,this was hands down the worse book i have ever...,1,this was hands down the worse book i have ever...,this was hands down the worse book i have ever...,-0.4767


In [14]:
bins = [-1, -0.1, 0.1, 1]
names = ['negative', 'neutral', 'positive']

In [15]:
data['sent_analyze_label'] = pd.cut(data['sent_analyze_score'], bins, labels=names)
data.head(5)

,index,reviewText,rating,reviewText_lowercase,reviewText_clean,sent_analyze_score,sent_analyze_label
0,11494,Clean and funny. A bit busy with all the diffe...,3,clean and funny. a bit busy with all the diffe...,clean and funny a bit busy with all the differ...,0.7684,positive
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,alex a sexy hot cop and the phd candidate. wha...,alex a sexy hot cop and the phd candidate what...,0.9325,positive
2,1463,Good thing that this is a free story. I read i...,1,good thing that this is a free story. i read i...,good thing that this is a free story i read it...,0.6740,positive
3,10342,"Action, action, action! Equipment keeps gettin...",4,"action, action, action! equipment keeps gettin...",action action action equipment keeps getting b...,0.6652,positive
4,5256,this was hands down the worse book i have ever...,1,this was hands down the worse book i have ever...,this was hands down the worse book i have ever...,-0.4767,negative


In [16]:
trans_analyze = pipeline('sentiment-analysis')

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [17]:
trans_analyze_label = []
for review in data['reviewText_clean']:
    sent_list = trans_analyze(review)
    sent_labels = [sent['label'] for sent in sent_list]
    trans_analyze_label.append(sent_labels)
data['transformers_analyze_label'] = trans_analyze_label
data.head(5)

,index,reviewText,rating,reviewText_lowercase,reviewText_clean,sent_analyze_score,sent_analyze_label,transformers_analyze_label
0,11494,Clean and funny. A bit busy with all the diffe...,3,clean and funny. a bit busy with all the diffe...,clean and funny a bit busy with all the differ...,0.7684,positive,[POSITIVE]
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4,alex a sexy hot cop and the phd candidate. wha...,alex a sexy hot cop and the phd candidate what...,0.9325,positive,[POSITIVE]
2,1463,Good thing that this is a free story. I read i...,1,good thing that this is a free story. i read i...,good thing that this is a free story i read it...,0.6740,positive,[NEGATIVE]
3,10342,"Action, action, action! Equipment keeps gettin...",4,"action, action, action! equipment keeps gettin...",action action action equipment keeps getting b...,0.6652,positive,[NEGATIVE]
4,5256,this was hands down the worse book i have ever...,1,this was hands down the worse book i have ever...,this was hands down the worse book i have ever...,-0.4767,negative,[NEGATIVE]
